# High-Dimensional Perceptron Memorization

This notebook studies a simple question: can a linear perceptron memorize random labels on real cat/dog images when each image has thousands of pixel features?

The short answer is yes, up to a point. A 64 x 64 grayscale image has 4096 pixel features. With a bias term, the linear classifier has 4097 adjustable degrees of freedom. That high dimensionality can allow perfect training-set fits even when the labels are random.

## Why Randomize The Labels?

The experiment starts with real cat and dog face images from AFHQ, but then replaces the true labels with random `-1` and `+1` labels.

That makes the interpretation clean:

- If the classifier fits the randomized labels, it is memorizing.
- It cannot be learning the visual concept of cat versus dog, because the labels no longer mean cat or dog.
- This is a compact way to demonstrate why perfect training accuracy can be misleading in high dimensions.

## Experiment Plan

1. Load AFHQ cat/dog images.
2. Resize each image to 64 x 64 grayscale.
3. Flatten each image into a 4096-dimensional vector.
4. Randomize the labels.
5. Test sample sizes below, at, and above `4097`.
6. Compare an exact separator proof with perceptron training.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from IPython.display import Image, display

from afhq_vc_demo.config import BIAS_DIM, IMAGE_SIZE, INPUT_DIM, KAGGLE_SLUG
from afhq_vc_demo.data import find_class_dirs, load_afhq_cat_dog, make_synthetic_vectors
from afhq_vc_demo.experiment import run_experiments
from afhq_vc_demo.reporting import write_outputs

DATA_ROOT = PROJECT_ROOT / "data" / "raw"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
SAMPLE_SIZES = [500, 1000, 2000, 4096, 4097, 5000]
SEED = 7
MAX_EPOCHS = 50

print(f"Dataset handle: {KAGGLE_SLUG}")
print(f"Image size: {IMAGE_SIZE} x {IMAGE_SIZE}")
print(f"Input dimension: {INPUT_DIM}; with bias: {BIAS_DIM}")

## Data Availability

The project expects AFHQ under `data/raw`. If it is not present, run this from the repository root:

```bash
python scripts/download_afhq.py --source kagglehub
```

If KaggleHub is unavailable, the script also supports `--source kaggle` and `--source official`.

In [ ]:
cat_dir, dog_dir = find_class_dirs(DATA_ROOT)
cat_count = sum(1 for p in cat_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
dog_count = sum(1 for p in dog_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print(f"Cat images available: {cat_count}")
print(f"Dog images available: {dog_count}")

## Load 64 x 64 Grayscale Vectors

The loader shuffles the images, converts them to grayscale, resizes them to 64 x 64, flattens them, and standardizes the pixel features.

In [ ]:
x, true_labels, paths = load_afhq_cat_dog(DATA_ROOT, max(SAMPLE_SIZES), SEED)
print(f"Feature matrix shape: {x.shape}")
print(f"Loaded examples: {len(paths)}")
print(f"True-label balance in loaded sample: cats={(true_labels == -1).sum()}, dogs={(true_labels == 1).sum()}")

## Run The Randomized-Label Experiment

The true labels are not used for training here. `run_experiments` creates a fixed random labeling from the seed and evaluates the same randomized split for each sample size.

In [ ]:
results = run_experiments(x, SAMPLE_SIZES, max_epochs=MAX_EPOCHS, seed=SEED)
write_outputs(results, OUTPUT_ROOT, source_name="Kaggle AFHQ dataset (andrewmvd/animal-faces), local data/raw")

for result in results:
    print(result)

## Results Table

In [ ]:
import pandas as pd

df = pd.DataFrame([r.__dict__ for r in results])
df

## Figures

The experiment writes reusable figures to `outputs/figures/`. They are linked here so the notebook stays lightweight while the visuals remain visible in GitHub and Jupyter.

### Training Error

![Training error](../outputs/figures/training_error.png)

### Rank Versus Sample Size

![Rank versus sample size](../outputs/figures/rank_vs_sample_size.png)

### Perceptron Updates In The Short Run

![Perceptron updates](../outputs/figures/perceptron_updates.png)

### Long-Run Convergence Epochs

![Long-run convergence epochs](../outputs/figures/perceptron_long_run_epochs.png)

### Weight Snapshots During Training

![Perceptron training journey](../outputs/figures/perceptron_training_journey_n500.png)

### Random Labels On Real Images

![Random label examples](../outputs/figures/random_label_examples_n500.png)

### How The Weights Accumulate Image-Like Updates

![Weight update story](../outputs/figures/weight_update_story_n500.png)

## What The Weight Images Mean

Each pixel has its own weight. Because the perceptron update is `w <- w + y_i*x_i`, the learned weight vector becomes a sum of signed training images. That is why the weights can be reshaped into a 64 x 64 heatmap.

In this notebook the labels are random, so a face-like heatmap is not evidence of cat/dog understanding. It is evidence that image-shaped examples were added and subtracted until the arbitrary labels were separated.

## Can We Predict The Epoch Count?

The VC-dimension argument tells us when a separator can exist. It does not give the exact number of epochs the perceptron will need.

The classic perceptron mistake bound says the number of mistakes is at most `(R / gamma)^2`, where `R` bounds the input norms and `gamma` is the separating margin. Small margins can make convergence slow, so the epoch count is best read as an empirical result for this specific subset, label assignment, feature scaling, and training order.

## How To Read This Result

This notebook is proving a chain of ideas. First, for `N <= 4097`, the bias-augmented image matrix has full row rank. That means an exact linear separator exists for any `-1`/`+1` labels on those images. The true cat/dog labels are one possible label vector, so they can be separated.

Second, the experiment uses randomized labels. If random labels can be separated, the separator cannot be learning cats versus dogs. It is memorizing an arbitrary assignment.

Third, the perceptron convergence theorem says that when data are linearly separable, the standard perceptron learning rule eventually converges after a finite number of updates. The 50-epoch training trace is a practical demo, not the formal proof; the rank and exact-construction check is the proof that the separable cases exist.

At `N = 5000`, the rank is still capped at 4097 and the exact separator proof no longer fits every random label. That is the contrast this experiment is designed to show.

## Demonstrating "Given Enough Iterations"

The short sweep used a 50-epoch training budget. That is useful because it shows practical training difficulty, but it is not the same thing as the perceptron convergence theorem.

To make the learning-rule claim concrete, longer perceptron runs were performed for separable randomized-label subsets. These runs use the actual Rosenblatt perceptron update rule and continue until zero training error or the larger epoch budget is exhausted.

In [ ]:
long_run = pd.read_csv(OUTPUT_ROOT / "tables" / "perceptron_long_run.csv")
long_run

These longer runs are the practical demonstration: the perceptron learning rule itself reaches perfect training separation on randomized labels for `N = 500`, `N = 1000`, and `N = 2000`.

The rank proof explains why this is possible. The perceptron convergence theorem explains why a separating hyperplane implies eventual convergence. The longer runs show that theorem in action on this image dataset.

## Small Synthetic Smoke Test

This tiny synthetic run is included only to verify the code path quickly. The real result above uses AFHQ images.

In [ ]:
SMOKE_SAMPLE_SIZES = [20, 40, 80]
x_synth, _ = make_synthetic_vectors(max(SMOKE_SAMPLE_SIZES), SEED)
synthetic_results = run_experiments(x_synth, SMOKE_SAMPLE_SIZES, max_epochs=10, seed=SEED)
for result in synthetic_results:
    print(result)